# K-mer Feature Pipeline (Binary presence)

This notebook performs k-mer feature selection and trains models using binary presence/absence
features for selection and final modeling. The prevalence-first approach keeps memory use low.
Models and vocabulary are saved to
`output/.`

In [3]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split


## Helper functions
These helpers iterate per-genome k-mer dump files, build prevalence counts, construct
sparse matrices (binary mode), and load phenotype labels.

In [9]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
from typing import Iterable

def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open("r", encoding="utf8", errors="ignore") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers


def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence: Counter = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f"{gid}_db_kmers.txt"
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        prevalence.update(kmers.keys())
    return prevalence


def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                               min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab


def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = True,
    chunk_size: int = 100,
) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `np.int32` for counts).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        rows: list[int] = []
        cols: list[int] = []
        data: list[int] = []
        for row_idx, gid in enumerate(chunk_ids):
            dump_path = dump_dir / f"{gid}_db_kmers.txt"
            if not dump_path.exists():
                continue
            kmers = iter_genome_kmers(dump_path)
            for kmer in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(kmers.get(kmer, 0)))
        if rows:
            block = sparse.csr_matrix((data, (rows, cols)), shape=(len(chunk_ids), n_features), dtype=dtype)
        else:
            block = sparse.csr_matrix((len(chunk_ids), n_features), dtype=dtype)
        blocks.append(block)
    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype)
    return sparse.vstack(blocks, format='csr')


def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `Genome ID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)

    # Normalize column lookup so small header variations do not break the notebook.
    norm = {c.strip().lower(): c for c in df.columns}
    gid_col = norm.get("genome id")
    pheno_col = norm.get("phenotype")

    if gid_col is None or pheno_col is None:
        raise ValueError(
            "Expected columns for Genome ID and phenotype in labels file. "
            f"Found columns: {list(df.columns)}"
        )

    pheno = df.set_index(gid_col)[pheno_col]
    pheno = pd.to_numeric(pheno, errors="coerce")
    return pheno

In [3]:

# def build_sparse_matrix(dump_dir: str, genome_ids: list[str], vocab: list[str], binary: bool = True) -> sparse.csr_matrix:
#     """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

#     This reads each genome's dump and populates the matrix using the provided `vocab` index.
#     When `binary` is True, presence is recorded as 1; otherwise counts are used (int).

#     Parameters:
#     - dump_dir: directory with per-genome k-mer dump files.
#     - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
#     - vocab: ordered list of k-mers corresponding to columns in the matrix.
#     - binary: whether to collapse counts to binary presence/absence.

#     Returns:
#     - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `int` values if counts).
#     """
#     vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
#     rows = []
#     cols = []
#     data = []
#     dump_dir = Path(dump_dir)
#     for row_idx, gid in enumerate(genome_ids):
#         dump_path = dump_dir / f'{gid}_db_kmers.txt'
#         if not dump_path.exists():
#             continue
#         kmers = iter_genome_kmers(dump_path)
#         for kmer in kmers.keys():
#             col_idx = vocab_index.get(kmer)
#             if col_idx is None:
#                 continue
#             rows.append(row_idx)
#             cols.append(col_idx)
#             data.append(1 if binary else int(kmers.get(kmer, 0)))
#     mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(genome_ids), len(vocab)), dtype=np.int8)
#     return mat

## Run Feature selection and Train models
Adjust the paths below (`dump_dir`, `labels_path`, `genome_ids_path`) if your files are elsewhere, then run this cell.

In [ ]:
# # define Paths
# dump_dir = Path('../data/counted_kmers')
# # expects columns: GenomeID, phenotype (R/S) and would have to adjust per antibiotics
# labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')  
# genome_ids_path = labels_path


In [ ]:
# # Load genome ids and labels
# genome_ids = []
# if genome_ids_path.exists():
#     id =pd.read_csv(genome_ids_path)
#     genome_ids = id['Genome ID'].tolist()

# y = load_labels(labels_path)
# # genome_ids = [gid for gid in genome_ids if gid in y.index]
# print(f'Using {len(genome_ids)} genomes for selection')


Using 5614 genomes for selection


#### Applying Prevalence Filtering

In [ ]:
# # Prevalence counting
# prevalence = build_prevalence(dump_dir, genome_ids)
# print(f'Unique k-mers observed: {len(prevalence):,d}')

Unique k-mers observed: 524,792


In [ ]:
##This code snippet is applying a prevalence filter to a set of k-mers (short DNA/protein sequences) based on their frequency across genomes. The comments explain that `min_frac=0.2` means that only k-mers present in at least 20% of genomes will be kept, while `max_frac=0.95` means that k-mers present in more than 95% of genomes will be excluded.
# # Prevalence filter, mmin_frac=0.2 means we keep k-mers present in at least 20% of genomes, 
# # max_frac=0.95 means we exclude k-mers present in more than 95% of genomes. This helps remove very rare k-mers (which may be noise) 
# # and very common k-mers (which may not be informative for classification).

# # the value of min_frac and max_frac is dependent on the number of genome present
# vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(genome_ids), min_frac=0.2, max_frac=0.95)
# print(f'Vocab after prevalence filter: {len(vocab):,d}')

Vocab after prevalence filter: 490,243


#### Build Sparse Matrix

In [ ]:
# # Build binary sparse matrix for selection
# X_bin = build_sparse_matrix(dump_dir, genome_ids, vocab, binary=True)
# y_arr = y.loc[genome_ids].to_numpy()
# print('Built binary matrix for selection:', X_bin.shape)

##### Applying Chi Square selection

In [ ]:
# # Chi-square selection
# top_k = min(100000, X_bin.shape[1])
# scores, _ = chi2(X_bin, y_arr)
# top_idx = np.argsort(scores)[::-1][:top_k]
# vocab_sel = [vocab[i] for i in top_idx]
# print(f'Selected top k-mers: {len(vocab_sel):,d}')


### Build final binary sparse data and Train model

In [ ]:
# # Build final binary matrix for modeling (only selected features)
# X_final = build_sparse_matrix(dump_dir, genome_ids, vocab_sel, binary=True)
# print('Built final binary matrix:', X_final.shape)


In [ ]:
# # Train and evaluate
# X_train, X_test, y_train, y_test = train_test_split(X_final, y_arr, test_size=0.2, random_state=42, stratify=y_arr)
# log = LogisticRegression(max_iter=1000, solver='saga')
# log.fit(X_train, y_train)
# y_pred = log.predict(X_test)
# print('Logistic accuracy (binary):', accuracy_score(y_test, y_pred))

# rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
# rf.fit(X_train, y_train)
# y_pred_rf = rf.predict(X_test)
# print('RF accuracy (binary):', accuracy_score(y_test, y_pred_rf))

# # 7) Save artifacts
# model_dir = Path('output/models')
# out_dir = Path('output/feature_selection')
# # out_dir.mkdir(parents=True, exist_ok=True)
# joblib.dump(vocab_sel, out_dir / 'vocab_selected_binary.pkl')
# joblib.dump(log, model_dir / 'logistic_binary.joblib')
# joblib.dump(rf, model_dir / 'rf_binary.joblib')
# with open(out_dir / 'results_binary.json', 'w') as fh:
#     json.dump({'logistic_acc': accuracy_score(y_test, y_pred), 'rf_acc': accuracy_score(y_test, y_pred_rf)}, fh, indent=2)
# print('Saved selected vocabulary and models to:  output/')

In [8]:
# === Sample 500 Resistant + 500 Susceptible and build sparse matrix ===
seed = 42
from pathlib import Path
import random

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels = load_labels(labels_path)
labels = labels.dropna()
labels = labels.astype(float)

res_ids = labels[labels == 1.0].index.astype(str).tolist()
sus_ids = labels[labels == 0.0].index.astype(str).tolist()

n_per_class = 500
if len(res_ids) < n_per_class or len(sus_ids) < n_per_class:
    raise ValueError(f'Not enough genomes to sample: have {len(res_ids)} R, {len(sus_ids)} S')

random.seed(seed)
sampled_res = random.sample(res_ids, n_per_class)
sampled_sus = random.sample(sus_ids, n_per_class)
sampled_ids = sampled_res + sampled_sus

out_ids_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
out_ids_path.parent.mkdir(parents=True, exist_ok=True)
with out_ids_path.open('w') as fh:
    fh.write('\n'.join(sampled_ids))

print(f'Sampled {len(sampled_ids)} genomes (R={n_per_class}, S={n_per_class}), saved to {out_ids_path}')


Sampled 1000 genomes (R=500, S=500), saved to ..\data\phenotype\ampicillin_1000_ids.txt


In [11]:

# Build prevalence and vocabulary (prevalence filter)
dump_dir = Path('../data/counted_kmers')
prevalence = build_prevalence(dump_dir, sampled_ids)
print('Unique k-mers seen:', len(prevalence))


Unique k-mers seen: 524689


In [12]:

# Filter by prevalence (2% - 95%) and cap vocab size if too large
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(sampled_ids), min_frac=0.02, max_frac=0.95)
print('Vocab after prevalence filter:', len(vocab))


Vocab after prevalence filter: 519756


In [ ]:

# max_vocab = 100000
# if len(vocab) > max_vocab:
#     # keep top by prevalence
#     vocab = [k for k, _ in prevalence.most_common(max_vocab) if k in set(vocab)]
#     print(f'Vocab capped to top {max_vocab} by prevalence -> {len(vocab)}')


In [17]:
# Build binary presence sparse matrix (rows ordered as sampled_ids)
X = build_sparse_matrix(dump_dir, sampled_ids, vocab, binary=True, chunk_size=100)
print('Built X shape:', X.shape)

# Build label vector aligned to sampled_ids
# Ensure labels index is string-typed to match sampled_ids
labels_str = labels.copy()
labels_str.index = labels_str.index.astype(str)

missing = [gid for gid in sampled_ids if gid not in labels_str.index]
if missing:
    raise KeyError(f'Some sampled ids are missing in labels: {missing[:5]}... total {len(missing)}')

y = np.array([labels_str.loc[gid] for gid in sampled_ids], dtype=int)
print('Built y shape:', y.shape)

# Save sampled ids and vocab (raw) for reproducibility
fv_dir = Path('../output/feature_selection')
fv_dir.mkdir(parents=True, exist_ok=True)
with (fv_dir / 'ampicillin_1000_sampled_ids.txt').open('w') as fh:
    fh.write('\n'.join(sampled_ids))
with (fv_dir / 'ampicillin_1000_vocab_raw.txt').open('w', encoding='utf8') as fh:
    fh.write('\n'.join(vocab))
print('Saved sampled ids and raw vocab.')

Built X shape: (1000, 519756)
Built y shape: (1000,)
Saved sampled ids and raw vocab.


In [18]:
# === Train / Evaluate / Save artifacts ===
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, balanced_accuracy_score, average_precision_score, roc_auc_score
import joblib

seed = 42
ids = np.array(sampled_ids)
idx = np.arange(len(ids))
train_idx, test_idx, y_train, y_test = train_test_split(idx, y, stratify=y, test_size=0.2, random_state=seed)
X_train = X[train_idx]
X_test = X[test_idx]
print('Train/test shapes:', X_train.shape, X_test.shape)


Train/test shapes: (800, 519756) (200, 519756)


In [44]:

# Feature selection + dimension reduction + classifier pipeline
k_features = 15000
# svd_components = 200

pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=min(k_features, X.shape[1]))),
    # ('svd', TruncatedSVD(n_components=min(svd_components, min(k_features, X.shape[1]) - 1), random_state=seed)),
    ('lr', LogisticRegression(solver='saga', class_weight='balanced', max_iter=1000, n_jobs=-1))
])

# lr = LogisticRegression(solver='saga', class_weight='balanced', max_iter=1000, n_jobs=-1)

In [ ]:
# #using Lazy Predictor to quickly benchmark multiple models but works only on pd dataframes
# from lazypredict.Supervised import LazyClassifier
# lazy_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
# models, predictions = lazy_clf.fit(X_train, X_test, y_train, y_test)
# print(models)

In [45]:

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
try:
    y_proba = pipeline.predict_proba(X_test)[:, 1]
except Exception:
    y_proba = None

print('Balanced Accuracy:', balanced_accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
if y_proba is not None:
    print('Average Precision (PR-AUC):', average_precision_score(y_test, y_proba))
    try:
        print('ROC AUC:', roc_auc_score(y_test, y_proba))
    except Exception:
        pass

# Save model and selected k-mers
out_models = Path('../output/models')
out_models.mkdir(parents=True, exist_ok=True)
model_path = out_models / 'ampicillin_pilot3.1_logreg.joblib'
joblib.dump(pipeline, model_path)
print('Saved model to', model_path)


Balanced Accuracy: 0.73
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.77      0.74       100
           1       0.75      0.69      0.72       100

    accuracy                           0.73       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.73      0.73      0.73       200

Average Precision (PR-AUC): 0.841650173144797
ROC AUC: 0.80725
Saved model to ..\output\models\ampicillin_pilot3.1_logreg.joblib


In [ ]:
# Train a linear SVM baseline on the same sparse k-mer features
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0, class_weight='balanced', random_state=seed)
svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)
svm_scores = svm.decision_function(X_test)

print('Linear SVM Balanced Accuracy:', balanced_accuracy_score(y_test, svm_pred))
print('Linear SVM Classification Report:\n', classification_report(y_test, svm_pred))
try:
    print('Linear SVM Average Precision (PR-AUC):', average_precision_score(y_test, svm_scores))
except Exception:
    pass
try:
    print('Linear SVM ROC AUC:', roc_auc_score(y_test, svm_scores))
except Exception:
    pass

# Save model and predictions
svm_path = out_models / 'ampicillin_pilot_linear_svm.joblib'
joblib.dump(svm, svm_path)
print('Saved Linear SVM to', svm_path)

pred_df_svm = pd.DataFrame({
    'GenomeID': ids[test_idx].astype(str),
    'y_true': y_test,
    'y_pred': svm_pred,
    'score': svm_scores,
})
pred_df_svm.to_csv(out_models / 'ampicillin_pilot_linear_svm_test_predictions.csv', index=False)
print('Saved Linear SVM test predictions to', out_models / 'ampicillin_pilot_linear_svm_test_predictions.csv')

In [22]:

# # Save selected k-mers
# selected_idx = lr.named_steps['selectk'].get_support(indices=True)
# selected_kmers = [vocab[i] for i in selected_idx]
# with (fv_dir / 'ampicillin_1000_selected_kmers.txt').open('w', encoding='utf8') as fh:
#     fh.write('\n'.join(selected_kmers))
# print('Saved selected k-mers:', len(selected_kmers))

# Save test predictions (with Genome IDs)
pred_df = pd.DataFrame({
    'GenomeID': ids[test_idx].astype(str),
    'y_true': y_test,
    'y_pred': y_pred,
})
pred_df.to_csv(out_models / 'ampicillin_pilot_test_predictions.csv', index=False)
print('Saved test predictions to', out_models / 'ampicillin_pilot_test_predictions.csv')

AttributeError: 'LogisticRegression' object has no attribute 'named_steps'

## Testing model on Held out dataset

In [14]:
# Evaluate held-out genomes (not in the 1000 sample)
# Builds features for the held-out genomes, runs the saved model, computes metrics, and saves predictions.
from sklearn.metrics import classification_report, balanced_accuracy_score, average_precision_score, roc_auc_score

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels_all = load_labels(labels_path).dropna()

# load sampled ids
sampled_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
if sampled_path.exists():
    sampled_ids_file = [s.strip() for s in sampled_path.read_text(encoding='utf8').splitlines() if s.strip()]
else:
    # fall back to in-memory variable if present
    sampled_ids_file = sampled_ids if 'sampled_ids' in globals() else []

# Determine held-out IDs (strings) that have labels
held_ids = [str(g) for g in labels_all.index.astype(str) if str(g) not in set(sampled_ids_file)]
print(f'Total labeled genomes: {len(labels_all)}, held-out candidates: {len(held_ids)}')

# Check for available k-mer dumps and filter
cand_dump_dirs = [Path('../output/counted_kmers'), Path('../data/counted_kmers')]
for d in cand_dump_dirs:
    if d.exists():
        dump_dir = d
        break
else:
    raise FileNotFoundError('Could not find counted_kmers directory in expected locations')

held_ids_with_dump = [gid for gid in held_ids if (dump_dir / f'{gid}_db_kmers.txt').exists()]
print(f'Held-out genomes with dumps: {len(held_ids_with_dump)}')
if not held_ids_with_dump:
    raise RuntimeError('No held-out genome dump files found; cannot evaluate')

# Load saved model
model_path = Path('../output/models/ampicillin_pilot_logreg.joblib')
if not model_path.exists():
    raise FileNotFoundError(f'Model not found at {model_path}')
model = joblib.load(model_path)

# Determine which vocab to use to build features
fv_dir = Path('../output/feature_selection')
raw_vocab_path = fv_dir / 'ampicillin_1000_vocab_raw.txt'
sel_vocab_path = fv_dir / 'ampicillin_1000_selected_kmers.txt'
if raw_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in raw_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using raw vocab (pre-selection) with', len(vocab_for_model), 'k-mers')
elif sel_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in sel_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using selected k-mers vocab with', len(vocab_for_model), 'k-mers')
else:
    # fallback: attempt to infer from saved model
    if hasattr(model, 'named_steps') and 'selectk' in model.named_steps:
        raise FileNotFoundError('Raw vocab required by pipeline but not found in feature_selection folder')
    else:
        # If model is a plain classifier, attempt to use selected_kmers if present
        vocab_for_model = []

# If we have a vocab, build sparse matrix; otherwise try to error with guidance
if vocab_for_model:
    X_held = build_sparse_matrix(dump_dir, held_ids_with_dump, vocab_for_model, binary=True, chunk_size=100)
else:
    raise RuntimeError('No vocabulary available to construct held-out feature matrix')

# Align labels
y_held = np.array([labels_all.loc[gid] for gid in held_ids_with_dump], dtype=int)

# If model is a pipeline, call predict/predict_proba directly. If it's a bare classifier, ensure feature dims match.
try:
    y_pred = model.predict(X_held)
except Exception as e:
    # If model expects dense input or different shape, try converting
    try:
        y_pred = model.predict(X_held.toarray())
    except Exception:
        raise

try:
    y_proba = model.predict_proba(X_held)[:, 1]
except Exception:
    try:
        y_proba = model.predict_proba(X_held.toarray())[:, 1]
    except Exception:
        y_proba = None

print('Held-out Balanced Accuracy:', balanced_accuracy_score(y_held, y_pred))
print('Held-out Classification Report:\n', classification_report(y_held, y_pred))
if y_proba is not None:
    print('Held-out Average Precision (PR-AUC):', average_precision_score(y_held, y_proba))
    try:
        print('Held-out ROC AUC:', roc_auc_score(y_held, y_proba))
    except Exception:
        pass

# Save predictions
out_models = Path('../output/models')
out_models.mkdir(parents=True, exist_ok=True)
pred_df = pd.DataFrame({
    'GenomeID': held_ids_with_dump,
    'y_true': y_held,
    'y_pred': y_pred,
})
if y_proba is not None:
    pred_df['y_proba'] = y_proba

pred_file = out_models / 'ampicillin_heldout_predictions.csv'
pred_df.to_csv(pred_file, index=False)
print('Saved held-out predictions to', pred_file)

# Save a short summary
summary = {
    'held_count': int(len(held_ids_with_dump)),
    'balanced_accuracy': float(balanced_accuracy_score(y_held, y_pred)),
}
with open(out_models / 'ampicillin_heldout_summary.json', 'w') as fh:
    json.dump(summary, fh)
print('Saved held-out summary')

Total labeled genomes: 5614, held-out candidates: 4614
Held-out genomes with dumps: 4386


FileNotFoundError: Model not found at ..\output\models\ampicillin_pilot_logreg.joblib